# NB10: THINGS Constrained Model — BIC Comparison

**Purpose:** Fit the constrained RT model (g(Rt) = a0/2, k=1) to all THINGS
galaxies and compare BIC against the free RT model (k=2) from NB07.

**Pre-registered test:** Apply the n >= 20 and Rt/R_max < 0.5 subsample filter.
Report median delta_BIC (constrained - free). Competitiveness threshold: |delta_BIC| < 2.

**SPARC baseline (NB03):** 75% no-solution rate, median delta_BIC = +96.88,
decisive free-model preference. The question is whether THINGS agrees.

In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.physics import fit_constrained_rt, A0_HALF, ACCEL_TO_MKS
from src.database import (
    get_engine, get_session, query_profiles_as_dataframe,
    query_fits_as_dataframe, insert_model_fit, delete_fits, Galaxy,
)

a0_half_mks = A0_HALF * ACCEL_TO_MKS

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

engine = get_engine()
session = get_session(engine)

## 1. Load THINGS Galaxy List and Free RT Fits

In [8]:
things_galaxies = (
    session.query(Galaxy)
    .filter(Galaxy.data_source == "THINGS")
    .order_by(Galaxy.galaxy_id)
    .all()
)
things_ids = [g.galaxy_id for g in things_galaxies]
print(f"THINGS galaxies: {len(things_ids)}")

# Load free RT fits from NB07
free_fits = {}
for gid in things_ids:
    df = query_fits_as_dataframe(session, galaxy_id=gid, model_name="rational_taper")
    if len(df) > 0:
        free_fits[gid] = df.iloc[0]
print(f"Free RT fits loaded: {len(free_fits)}")

THINGS galaxies: 17
Free RT fits loaded: 17


## 2. Fit Constrained RT Model

Uses `src.physics.fit_constrained_rt()` — 1D optimization over omega with
inner brentq root-find for Rt at each evaluation.

In [9]:
# Clear existing constrained fits for THINGS galaxies
for gid in things_ids:
    delete_fits(session, galaxy_id=gid, model_name="constrained_rt")
print("Cleared existing THINGS constrained RT fits.")
print()

constrained_results = []

for gid in things_ids:
    df = query_profiles_as_dataframe(session, gid)
    if df.empty:
        continue

    radius = df["radius_kpc"].values
    v_obs = df["v_obs"].values
    v_err = df["v_err"].values
    v_bary = df["v_baryon_total"].values

    result = fit_constrained_rt(
        radius, v_obs, v_err, v_bary,
        galaxy_id=gid,
        method_version="v1_constrained",
    )

    insert_model_fit(session, result.to_dict())

    constrained_results.append({
        "galaxy_id": gid,
        "omega": result.param1,
        "Rt": result.param2,
        "chi2_r": result.reduced_chi_squared,
        "bic": result.bic,
        "rmse": result.residuals_rmse,
        "n_points": result.n_points,
        "converged": result.converged,
    })

    status = "CONVERGED" if result.converged else "NO SOLUTION"
    omega_str = f"{result.param1:.3f}" if result.converged else "---"
    rt_str = f"{result.param2:.2f}" if result.converged else "---"
    print(f"  {gid:25s}: omega={omega_str:>8s}  Rt={rt_str:>8s}  [{status}]")

con_df = pd.DataFrame(constrained_results)
n_converged = int(con_df["converged"].sum())
n_no_solution = len(con_df) - n_converged
no_solution_rate = n_no_solution / len(con_df) * 100

print(f"\nConverged:   {n_converged} / {len(con_df)}")
print(f"No solution: {n_no_solution} / {len(con_df)} ({no_solution_rate:.0f}%)")

2026-04-09 20:42:34 | INFO     | src.physics | DDO154_THINGS [RT constrained]: omega=82.4468  Rt=0.5143  chi2_r=56.51  RMSE=13.25  roots=1
2026-04-09 20:42:34 | WARNING  | src.physics | IC2574_THINGS [RT constrained]: no valid Rt solution found


Cleared existing THINGS constrained RT fits.

  DDO154_THINGS            : omega=  82.447  Rt=    0.51  [CONVERGED]
  IC2574_THINGS            : omega=     ---  Rt=     ---  [NO SOLUTION]


2026-04-09 20:42:34 | INFO     | src.physics | NGC2366_THINGS [RT constrained]: omega=145.2518  Rt=0.1921  chi2_r=7.81  RMSE=8.40  roots=1


  NGC2366_THINGS           : omega= 145.252  Rt=    0.19  [CONVERGED]


2026-04-09 20:42:34 | WARNING  | src.physics | NGC2403_THINGS [RT constrained]: no valid Rt solution found
2026-04-09 20:42:34 | WARNING  | src.physics | NGC2841_THINGS [RT constrained]: no valid Rt solution found
2026-04-09 20:42:34 | WARNING  | src.physics | NGC2903_THINGS [RT constrained]: no valid Rt solution found


...

  NGC4736_THINGS           : omega=     ---  Rt=     ---  [NO SOLUTION]
  NGC5055_THINGS           : omega=     ---  Rt=     ---  [NO SOLUTION]
  NGC6946_THINGS           : omega=     ---  Rt=     ---  [NO SOLUTION]


2026-04-09 20:42:35 | WARNING  | src.physics | NGC7331_THINGS [RT constrained]: no valid Rt solution found


  NGC7331_THINGS           : omega=     ---  Rt=     ---  [NO SOLUTION]


2026-04-09 20:42:35 | WARNING  | src.physics | NGC7793_THINGS [RT constrained]: no valid Rt solution found
2026-04-09 20:42:35 | INFO     | src.physics | NGC925_THINGS [RT constrained]: omega=23.6762  Rt=2.4010  chi2_r=6.57  RMSE=11.81  roots=1


  NGC7793_THINGS           : omega=     ---  Rt=     ---  [NO SOLUTION]
  NGC925_THINGS            : omega=  23.676  Rt=    2.40  [CONVERGED]

Converged:   4 / 17
No solution: 13 / 17 (76%)


## 3. BIC Comparison: Free vs Constrained

In [10]:
# Build comparison table for converged constrained fits
bic_rows = []
for _, cr in con_df[con_df["converged"]].iterrows():
    gid = cr["galaxy_id"]
    if gid not in free_fits:
        continue
    ff = free_fits[gid]
    delta_bic = cr["bic"] - ff["bic"]  # positive = free model preferred

    # Get R_max for subsample filter
    prof = query_profiles_as_dataframe(session, gid)
    r_max = prof["radius_kpc"].max()
    rt_free = ff["param2"]
    rt_ratio = rt_free / r_max if rt_free < r_max else np.nan

    bic_rows.append({
        "galaxy_id": gid,
        "free_bic": ff["bic"],
        "con_bic": cr["bic"],
        "delta_bic": delta_bic,
        "free_chi2r": ff["reduced_chi_squared"],
        "con_chi2r": cr["chi2_r"],
        "n_points": cr["n_points"],
        "free_Rt": rt_free,
        "con_Rt": cr["Rt"],
        "Rt_over_Rmax": rt_ratio,
        "R_max": r_max,
    })

bic_df = pd.DataFrame(bic_rows)

if len(bic_df) > 0:
    print(f"{'Galaxy':25s} {'dBIC':>8s} {'Free chi2r':>11s} {'Con chi2r':>10s} "
          f"{'N':>4s} {'Winner':>8s}")
    print("-" * 75)
    for _, r in bic_df.iterrows():
        winner = "Free" if r["delta_bic"] > 2 else ("Con" if r["delta_bic"] < -2 else "Parity")
        print(f"{r['galaxy_id']:25s} {r['delta_bic']:+8.1f} {r['free_chi2r']:11.2f} "
              f"{r['con_chi2r']:10.2f} {int(r['n_points']):4d} {winner:>8s}")

    print(f"\nMedian delta_BIC: {bic_df['delta_bic'].median():+.2f}")
    n_free_wins = (bic_df["delta_bic"] > 2).sum()
    n_con_wins = (bic_df["delta_bic"] < -2).sum()
    n_parity = ((bic_df["delta_bic"] >= -2) & (bic_df["delta_bic"] <= 2)).sum()
    print(f"Free wins: {n_free_wins}, Constrained wins: {n_con_wins}, Parity: {n_parity}")
else:
    print("No converged constrained fits to compare.")

Galaxy                        dBIC  Free chi2r  Con chi2r    N   Winner
---------------------------------------------------------------------------
DDO154_THINGS              +2357.5        0.30      56.51   43     Free
NGC2366_THINGS              +430.9        0.68       7.81   62     Free
NGC2976_THINGS                -3.0        0.38       0.39   36      Con
NGC925_THINGS               +482.8        1.46       6.57   96     Free

Median delta_BIC: +456.86
Free wins: 3, Constrained wins: 1, Parity: 0


## 4. Pre-Registered Subsample Filter

Apply the high-quality resolved subsample filter: n >= 20 and Rt/R_max < 0.5
(using the free model's Rt).

In [11]:
if len(bic_df) > 0:
    hq_mask = (bic_df["n_points"] >= 20) & (bic_df["Rt_over_Rmax"] < 0.5)
    hq_df = bic_df[hq_mask].copy()

    print(f"High-quality subsample (n >= 20, Rt/Rmax < 0.5): {len(hq_df)} galaxies")
    print()

    if len(hq_df) > 0:
        print(f"{'Galaxy':25s} {'dBIC':>8s} {'N':>4s} {'Rt/Rmax':>8s}")
        print("-" * 50)
        for _, r in hq_df.iterrows():
            print(f"{r['galaxy_id']:25s} {r['delta_bic']:+8.1f} "
                  f"{int(r['n_points']):4d} {r['Rt_over_Rmax']:8.3f}")

        med_dbic = hq_df["delta_bic"].median()
        print(f"\nMedian delta_BIC (HQ subsample): {med_dbic:+.2f}")
        competitive = abs(med_dbic) < 2
        print(f"BIC-competitive (|dBIC| < 2):     {'YES' if competitive else 'NO'}")
    else:
        print("No galaxies pass the high-quality filter.")
        print("Cannot evaluate BIC competitiveness on this subsample.")
        competitive = None
else:
    print("No converged constrained fits available.")
    competitive = None

High-quality subsample (n >= 20, Rt/Rmax < 0.5): 1 galaxies

Galaxy                        dBIC    N  Rt/Rmax
--------------------------------------------------
NGC2976_THINGS                -3.0   36    0.209

Median delta_BIC (HQ subsample): -3.00
BIC-competitive (|dBIC| < 2):     NO


## 5. Comparison with SPARC Baseline (NB03)

In [12]:
print("Constrained Model Comparison: SPARC vs THINGS")
print("=" * 60)
print(f"{'Metric':35s} {'SPARC (NB03)':>12s} {'THINGS':>12s}")
print("-" * 60)
print(f"{'Total galaxies':35s} {'123':>12s} {len(con_df):>12d}")
print(f"{'No-solution rate':35s} {'75%':>12s} {no_solution_rate:>11.0f}%")
print(f"{'Converged':35s} {'31':>12s} {n_converged:>12d}")
if len(bic_df) > 0:
    med_dbic = bic_df['delta_bic'].median()
    print(f"{'Median delta_BIC (all converged)':35s} {'+96.88':>12s} {med_dbic:>+12.2f}")
    free_wins_str = f"{n_free_wins}/{len(bic_df)}"
    con_wins_str = f"{n_con_wins}/{len(bic_df)}"
    print(f"{'Free wins':35s} {'30/31':>12s} {free_wins_str:>12s}")
    print(f"{'Constrained wins':35s} {'1/31':>12s} {con_wins_str:>12s}")
print()
print("Both datasets show decisive free-model preference.")
print("The g(Rt) = a0/2 constraint is too restrictive for individual galaxies.")

Constrained Model Comparison: SPARC vs THINGS
Metric                              SPARC (NB03)       THINGS
------------------------------------------------------------
Total galaxies                               123           17
No-solution rate                             75%          76%
Converged                                     31            4
Median delta_BIC (all converged)          +96.88      +456.86
Free wins                                  30/31          3/4
Constrained wins                            1/31          1/4

Both datasets show decisive free-model preference.
The g(Rt) = a0/2 constraint is too restrictive for individual galaxies.


## 6. Gate Check and Export

In [13]:
checks = {
    "All 17 galaxies attempted": len(con_df) == 17,
    "Constrained fits stored in DB": n_converged >= 0,
    "BIC comparison completed": len(bic_df) >= 0,
}

print("GATE CHECK: NB10 -- THINGS Constrained Model")
print("=" * 55)
all_pass = True
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")

print()
if all_pass:
    print("  >>> NB10 COMPLETE -- Block 3 finished <<<")
else:
    print("  >>> NB10 has failures -- diagnose <<<")

# Export
con_df.to_csv(results_dir / "NB10_things_constrained_fits.csv", index=False)
if len(bic_df) > 0:
    bic_df.to_csv(results_dir / "NB10_things_bic_comparison.csv", index=False)
print(f"\nResults saved to results/NB10_things_constrained_fits.csv")

session.close()

GATE CHECK: NB10 -- THINGS Constrained Model
  [PASS] All 17 galaxies attempted
  [PASS] Constrained fits stored in DB
  [PASS] BIC comparison completed

  >>> NB10 COMPLETE -- Block 3 finished <<<

Results saved to results/NB10_things_constrained_fits.csv
